# Improving Patch Classification
In this notebook, I'm going to explore how to improve the patch classifier as the current one is not really performing. In fact let's try out a bunch of the pretrained ones from torch vision. We have quite large datasets so it makes sense to try and train the whole network, not just the end layers.

In [ ]:
# standard project preamble
from pathgen.utils.seeds import set_seed
from pathgen.utils.paths import project_root

experiment_name = "all"
experiment_root = project_root() / "experiments" / experiment_name

global_seed = 987654321
set_seed(global_seed)

In [ ]:
# standard data science imports
import numpy as np

# standard pytorch imports
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim

# pytorch data loading imports
from torch.utils.data import DataLoader, RandomSampler
from torchvision.datasets import ImageFolder

# torchvision
import torchvision
import torchvision.transforms as transforms

In [ ]:
# define our hyper parameters
batch_size = 128
num_epochs = 30
learning_rate = 0.00001  # 0.001 - takes 25mins (log10 grid search?)

In [ ]:
# set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# set up the data loaders for training
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])  # these values are what the pretrained models expect
])

train_set = ImageFolder(experiment_root / 'train_patches', transform=transform)  # 70,000 samples (35,000 per class) - non-fake
valid_set = ImageFolder(experiment_root / 'valid_patches', transform=transform)  # 30,000 samples (15,000 per class) - non-fake

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, worker_init_fn=np.random.seed(global_seed), num_workers=32)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True, worker_init_fn=np.random.seed(global_seed), num_workers=32)

# downsample to speed up training to test the code
#train_sampler = RandomSampler(train_set, replacement=True, num_samples=256)
#valid_sampler = RandomSampler(valid_set, replacement=True, num_samples=256)

#train_loader = DataLoader(train_set, batch_size=batch_size, sampler=train_sampler, worker_init_fn=np.random.seed(global_seed), num_workers=32)
#valid_loader = DataLoader(valid_set, batch_size=batch_size, sampler=valid_sampler, worker_init_fn=np.random.seed(global_seed), num_workers=32)

## Saving and loading models
Let's add some functions for loading an saving the state of the model and training using a checkpoint.

In [ ]:
def save_checkpoint(epoch, model, optimizer, path):
    print(f"saving checkpoint to {path}")
    state = { 'epoch': epoch, 
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict() }
    torch.save(state, path)

def load_checkpoint(model, optimizer, path):
    print("loading checkpoint")
    state = torch.load(path)
    epoch = state["epoch"]
    model.load_state_dict(state["state_dict"])
    optimizer.load_state_dict(state["optimizer"])
    return epoch, model, optimizer, loss

## Set up the models that we are going to test
We are going to test using VGG16, Resnet-18, InceptionV3.

In [ ]:
# preamble
from torchsummary import summary

# get and example image from the dataset so we can generate the summaries
img, _ = train_set[0]
img.to(device)

In [ ]:
# VGG16 load the model
model_vgg16 = torchvision.models.vgg16(pretrained=False)
model_vgg16.classifier[6] = nn.Sequential(nn.Linear(in_features=4096, out_features=1, bias=True))

# get a summary of the model based on our input shape
model_vgg16.to(device)
summary(model_vgg16, img.shape, batch_size)

In [ ]:
# load in the pretrained resnet18 model and change the number of outputs on the final fc layer
model_resnet18 = torchvision.models.resnet18(pretrained=False)
model_resnet18.fc = nn.Linear(model_resnet18.fc.in_features, 1)

# get a summary of the model based on our input shape
model_resnet18.to(device)
print(model_resnet18)
summary(model_resnet18, img.shape, batch_size)

In [ ]:
# load in the pretrained densenet model and change the number of outputs on the final fc layer
model_googlenet = torchvision.models.googlenet(pretrained=True)
model_googlenet.fc = nn.Linear(model_googlenet.fc.in_features, 1)

# get a summary of the model based on our input shape
model_googlenet.to(device)
summary(model_googlenet, img.shape, batch_size)

## Training
Now that we have the data loader created and the model defined, let's define our loss and optimiser and run the training loop.

In [ ]:
from statistics import mean

class LoggedVariable:
    def __init__(self):
        self.batch_values = []
        self.epoch_values = []
        
    def append(self, value):
        self.batch_values.append(value)
        
    def end_epoch(self):
        mean_batch_values = mean(self.batch_values)
        self.epoch_values.append(mean_batch_values)
        self.batch_values = []
        
class Logger:
    def __init__(self):
        self.variables = {}
    
    def __call__(self, key, value):
        if key not in self.variables:
            self.variables[key] = LoggedVariable()
        self.variables[key].append(value)
    
    def end_epoch(self, epoch):
        print(f"end epoch {epoch}", end = '')
        for key, val in self.variables.items():
            val.end_epoch()
            print(f" {key}: {val.epoch_values[epoch]:.2f}", end = '')
        print()
    
    def history(self):
        return { k:v.epoch_values for k, v in self.variables.items() }

In [ ]:
def accuracy(scores, targets):
    _, predictions = scores.max(1)
    num_correct = (predictions == targets).sum()
    num_samples = predictions.size(0)
    return num_correct / num_samples

def binary_acc(y_pred, y_test):
    y_pred_tag = torch.round(torch.sigmoid(y_pred))
    correct_results_sum = (y_pred_tag == y_test).sum().float()
    acc = correct_results_sum/y_test.shape[0]
    acc = torch.round(acc * 100)
    return acc

In [ ]:
from time import time

def fit(model, optimizer_class, criterion, train_loader, valid_loader, device, epochs=20, learning_rate=0.001):
    print(f'fitting model: {type(model).__name__} for {epochs} epochs.')
    
    # set up the optimizer
    optimizer = optimizer_class(model.parameters(), lr=learning_rate)
    
    # initalise stats
    log = Logger()
    start_time_sec = time()
    
    for epoch in range(epochs):
        
        # train and evaluate on the training set
        model.train()
        for batch_idx, (data, targets) in enumerate(train_loader):
            # put X and y for the batch on the GPU is possible
            data = data.to(device=device)
            targets = targets.to(device=device)
            
            # forward pass
            scores = model(data)
            loss = criterion(scores, targets.unsqueeze(1).float())
            
            # backwards pass
            optimizer.zero_grad()
            loss.backward()
            
            # gradient descent
            optimizer.step()
            
            # log the metrics
            acc = binary_acc(scores, targets.unsqueeze(1))
            log('train_acc', acc.item())
            log('train_loss', loss.item())
            
            print('\r', f'train.\t\tepoch: {epoch}\tbatch: {batch_idx + 1}/{len(train_loader)}\tloss: {loss:.3f}\taccuracy: {acc} ', sep='', end='', flush=True)
        
        print()
        
        # evaluate on the validation set
        model.eval()
        for batch_idx, (data, targets) in enumerate(valid_loader):
            # put X and y for the batch on the device
            data = data.to(device=device)
            targets = targets.to(device=device)
            
            # compute the predictions
            scores = model(data)
            
            # computer the metric and log them
            loss = criterion(scores, targets.unsqueeze(1).float())
            acc = binary_acc(scores, targets.unsqueeze(1))
            log('valid_acc', acc.item())
            log('valid_loss', loss.item())
        
            print('\r', f'validate.\tepoch: {epoch}\tbatch: {batch_idx + 1}/{len(valid_loader)}\tloss: {loss:.3f}\taccuracy: {acc} ', sep='', end='', flush=True)        
        
        print()
        
        save_checkpoint(epoch, model, optimizer, experiment_root / f"{type(model).__name__}_checkpoint_{epoch}.ckpt") 
        log.end_epoch(epoch)
        
        print()
    
    end_time_sec       = time()
    total_time_sec     = end_time_sec - start_time_sec
    time_per_epoch_sec = total_time_sec / epochs
    print("training complete.")
    print('Time total:     %5.2f sec' % (total_time_sec))
    print('Time per epoch: %5.2f sec' % (time_per_epoch_sec))
    print()
    
    return log.history()
    

In [ ]:
# criterion = nn.BCEWithLogitsLoss()  # combines a Sigmoid layer and the BCELoss in one single class
# history = fit(model_vgg16, optim.Adam, criterion, train_loader, valid_loader, device, epochs=num_epochs, learning_rate=learning_rate)
# history

In [ ]:
import json

criterion = nn.BCEWithLogitsLoss()
# models = [model_vgg16, model_resnet18, model_googlenet]
models = [model_resnet18]
for model in models:
    # fit the model
    history = fit(model, optim.Adam, criterion, train_loader, valid_loader, device, epochs=num_epochs)
    
    # save the results
    json.dump(history, open(experiment_root / f"results_{type(model).__name__}.json", "w"))

Now let's visualise the training process and see how things went.

Let's test the model on the test set to see how it performs.